CNN Architectures

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Alexnet

class AlexNet(nn.Module):

    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()

        # ----------------------------------------------------
        # Convolutional layers
        # ----------------------------------------------------
        self.features = nn.Sequential(

            # Conv2D: 96 filters, 11x11 kernel, stride 4
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4),
            nn.ReLU(),

            # Max Pooling: 3x3, stride 2
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d( in_channels=96, out_channels=256, kernel_size=5 ),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3),
            nn.ReLU(),

            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3),
            nn.ReLU(),



            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3),
            nn.ReLU(),

            # Final Max Pooling
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        # Fully Connected Layers
        self.classifier = nn.Sequential(

            # Flatten
            nn.Flatten(),

            # Fully Connected Layer: 4096 neurons
            nn.Linear(256 * 2 * 2, 4096),
            nn.ReLU(),

            # Dropout 50%
            nn.Dropout(p=0.5),

            # Fully Connected Layer: 4096 neurons
            nn.Linear(4096, 4096),
            nn.ReLU(),

            nn.Dropout(p=0.5),

            # Output Layer: 10 classes
            nn.Linear(4096, num_classes),

            # Softmax
            nn.Softmax(dim=1)
        )


    def forward(self, x):

        # Pass through convolutional layers
        x = self.features(x)

        # Pass through fully connected layers
        x = self.classifier(x)

        return x

# Create the Alexnet Model

alexnet = AlexNet(num_classes=10)

# Print Model Summary

print("\n" + "*" * 60)
print("                ALEXNET MODEL SUMMARY")
print("*" * 60)

print(alexnet)

# Count trainable parameters
alexnet_params = sum(
    p.numel()
    for p in alexnet.parameters()
    if p.requires_grad
)

print(f"\nTotal Trainable Parameters: {alexnet_params:,}")


# Residual Block

class ResidualBlock(nn.Module):

    def __init__(self, filters=64):
        super(ResidualBlock, self).__init__()

        # First 3x3 convolution
        self.conv1 = nn.Conv2d(in_channels=filters, out_channels=filters, kernel_size=3, stride=1, padding=1)

        # Second 3x3 convolution
        self.conv2 = nn.Conv2d(in_channels=filters, out_channels=filters, kernel_size=3, stride=1, padding=1)


    def forward(self, input_tensor):

        # Save original input for skip connection
        identity = input_tensor

        # First convolution + ReLU
        x = self.conv1(input_tensor)
        x = F.relu(x)

        # Second convolution
        x = self.conv2(x)


        # Skip Connection and add original input to convolution output
        x = x + identity

        # ReLU after addition
        x = F.relu(x)

        return x


# Simple Resnet-Like Model


class SimpleResNet(nn.Module):

    def __init__(self, num_classes=10):
        super(SimpleResNet, self).__init__()

        # Initial Conv2D 64 filters 7x7 kernel stride = 2
        self.initial_conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3)

        # Two Residual Blocks
        self.residual_block1 = ResidualBlock(filters=64)

        self.residual_block2 = ResidualBlock(filters=64)

        # Flatten Layer
        self.flatten = nn.Flatten()

        """Input = 32x32
        After initial stride 2 convolution:
        32x32 -> 16x16
        Residual blocks maintain 16x16
        Flatten size:
        64 x 16 x 16 = 16384"""

        # Dense Layer: 128 neurons
        self.fc1 = nn.Linear(
            64 * 16 * 16,
            128
        )

        # Output Layer: 10 neurons
        self.fc2 = nn.Linear(
            128,
            num_classes
        )

        self.softmax = nn.Softmax(dim=1)


    def forward(self, x):

        # Initial convolution
        x = self.initial_conv(x)
        x = F.relu(x)

        # First residual block
        x = self.residual_block1(x)

        # Second residual block
        x = self.residual_block2(x)

        # Flatten
        x = self.flatten(x)

        # Dense layer
        x = self.fc1(x)
        x = F.relu(x)

        # Output layer
        x = self.fc2(x)

        # Softmax probabilities
        x = self.softmax(x)

        return x



# Create resnet model
resnet_model = SimpleResNet(num_classes=10)


# Print resnet Summary

print("\n\n" + "*" * 60)
print("              RESNET-LIKE MODEL SUMMARY")
print("*" * 60)

print(resnet_model)

# Count trainable parameters
resnet_params = sum(
    p.numel()
    for p in resnet_model.parameters()
    if p.requires_grad
)

print(f"\nTotal Trainable Parameters: {resnet_params:,}")




************************************************************
                ALEXNET MODEL SUMMARY
************************************************************
AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1))
    (9): ReLU()
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1))
    (11): ReLU()
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=4096, bias=True)
    (2): ReLU()
    (3): Dro